# Fig 7: TTFT Slowdown from Power-Capping (300W vs 200W)

**Requires sudo:** `sudo nvidia-smi -i <GPU_ID> -pl 200` to set the 200W power cap before running the collect cell.  
**Output:** `paper/figures/section3/output/<MODEL_SHORT>/prefill_powercap_ratio.{pdf,png}`

> ⚠️ `plot_prefill_powercap_ratio.py` reads from `paper/figures/section3/output/<MODEL_SHORT>/{300W,200W}/cache_cost_table.csv` (pre-reorganization paths, not yet updated to `model_outputs/`). The collect cell below writes 200W data to that same legacy location. 300W data must come from the same location — copy `cache_cost_table.csv` from `model_outputs/<MODEL_SHORT>/paper/section3/fig2/` if needed.

### Call order
1. Collect 300W cache-cost data (see Fig 2 notebook) and ensure `cache_cost_table.csv` is at `paper/figures/section3/output/<MODEL_SHORT>/300W/`
2. Collect 200W cache-cost data (collect cell below): `profiling/run_cache_cost.py --out output/<MODEL_SHORT>/200W/cache_cost_data` → `scripts/plot_cache_cost.py --dir output/<MODEL_SHORT>/200W`
3. `scripts/plot_prefill_powercap_ratio.py`

In [ ]:
import subprocess
from pathlib import Path

REPO_ROOT = next(
    p for p in [Path.cwd()] + list(Path.cwd().parents)
    if (p / ".conserve_root").exists()
)


def run(cmd):
    buf = []
    with subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    ) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
            buf.append(line)
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed (exit {proc.returncode}): {' '.join(str(c) for c in cmd)}")
    return "".join(buf)


In [ ]:
# Steps 1-2: collect raw cache-cost data at both power levels. Skip if data already exists.
# Note: 300W data is also required by Fig 2 — see that notebook to collect it.
# For 200W: set GPU power cap before running (requires sudo):
#   sudo nvidia-smi -i <GPU_ID> -pl 200
import sys; sys.path.insert(0, str(REPO_ROOT / "profiling"))
from config import MODEL_SHORT

out_200w = REPO_ROOT / "paper/figures/section3/output" / MODEL_SHORT / "200W" / "cache_cost_data"
run(["python", str(REPO_ROOT / "profiling/run_cache_cost.py"), "--out", str(out_200w)])
run([
    "python", str(REPO_ROOT / "paper/figures/section3/scripts/plot_cache_cost.py"),
    "--dir", str(REPO_ROOT / "paper/figures/section3/output" / MODEL_SHORT / "200W"),
])

In [ ]:
# Step 3: plot
%matplotlib inline
%run ../scripts/plot_prefill_powercap_ratio.py

In [ ]:
from IPython.display import Image
import sys; sys.path.insert(0, str(REPO_ROOT / "profiling"))
from config import MODEL_SHORT

Image(str(REPO_ROOT / "paper/figures/section3/output" / MODEL_SHORT / "prefill_powercap_ratio.png"))